In [ ]:
#Qué hace: discretiza el tiempo en ventanas de 24h relativas a T₀.
#Clave: asegura comparabilidad temporal entre pacientes.

In [1]:
# Imports

In [2]:
import pandas as pd
import numpy as np

df_base = pd.read_parquet("04_cohorte_base_T0.parquet")

In [3]:
# Crear rango de seguimiento por paciente

In [4]:
df_base["icu_out"] = pd.to_datetime(df_base["icu_out"])
df_base["t0_antibiotic"] = pd.to_datetime(df_base["t0_antibiotic"])

In [5]:
# Crear duración máxima permitida (censura a día 30)

In [6]:
df_base["max_followup"] = df_base["t0_antibiotic"] + pd.Timedelta(days=30)
df_base["followup_end"] = df_base[["icu_out", "max_followup"]].min(axis=1)

In [7]:
# Crear el número de días de seguimiento 

In [8]:
df_base["n_days"] = ((df_base["followup_end"] - df_base["t0_antibiotic"]).dt.total_seconds() // 86400).astype(int)
df_base.head()

,subject_id,hadm_id,icu_stay_id,infection_time,t0_antibiotic,organism,pathogen_group,site,age,gender,icu_in,icu_out,max_followup,followup_end,n_days
0,11239107,25883588,32367987,2113-01-16 17:03:00,2113-01-16 20:00:00,virus,Other,bronchoalveolar lavage,70,M,2113-01-15 16:00:00,2113-02-08 14:53:05,2113-02-15 20:00:00,2113-02-08 14:53:05,22
1,15811456,29271096,32876962,2144-06-26 20:32:00,2144-06-27 08:00:00,herpes simplex virus type 1,Other,bronchoalveolar lavage,64,F,2144-06-26 17:10:29,2144-07-01 12:20:13,2144-07-27 08:00:00,2144-07-01 12:20:13,4
2,19326831,29957742,32456504,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-08-10 10:23:03,2158-08-13 16:20:08,2158-08-21 07:00:00,2158-08-13 16:20:08,22
3,19326831,29957742,33950322,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-08-17 07:25:16,2158-08-19 20:15:15,2158-08-21 07:00:00,2158-08-19 20:15:15,28
4,19326831,29957742,38112378,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-07-22 07:12:49,2158-08-05 19:00:29,2158-08-21 07:00:00,2158-08-05 19:00:29,14


In [9]:
# Expandir filas → una fila por día y paciente

In [10]:
rows = []

for _, row in df_base.iterrows():
    for day in range(row["n_days"] + 1):  # incluye day=0
        window_start = row["t0_antibiotic"] + pd.Timedelta(days=day)
        window_end = window_start + pd.Timedelta(days=1)

        rows.append({
            "subject_id": row["subject_id"],
            "hadm_id": row["hadm_id"],
            "icu_stay_id": row["icu_stay_id"],
            "day_idx": day,
            "window_start": window_start,
            "window_end": window_end,
            "infection_time": row["infection_time"],
            "pathogen_group": row["pathogen_group"],
            "age": row["age"],
            "gender": row["gender"]
        })

df_windows = pd.DataFrame(rows)

In [11]:
# Guardar dataset longitudinal

In [12]:
df_windows.to_parquet("05_ventanas_24h.parquet", index=False)
df_windows.head()

,subject_id,hadm_id,icu_stay_id,day_idx,window_start,window_end,infection_time,pathogen_group,age,gender
0,11239107,25883588,32367987,0,2113-01-16 20:00:00,2113-01-17 20:00:00,2113-01-16 17:03:00,Other,70,M
1,11239107,25883588,32367987,1,2113-01-17 20:00:00,2113-01-18 20:00:00,2113-01-16 17:03:00,Other,70,M
2,11239107,25883588,32367987,2,2113-01-18 20:00:00,2113-01-19 20:00:00,2113-01-16 17:03:00,Other,70,M
3,11239107,25883588,32367987,3,2113-01-19 20:00:00,2113-01-20 20:00:00,2113-01-16 17:03:00,Other,70,M
4,11239107,25883588,32367987,4,2113-01-20 20:00:00,2113-01-21 20:00:00,2113-01-16 17:03:00,Other,70,M
